# Reproducing Hazon et al. (2022) from CRCNS hc-30: IMSE fits with the paper's repetitions

**Paper.** Hazon O, Minces VH, Tomàs DP, Ganguli S, Schnitzer MJ, Jercog PE (2022). Noise
correlations in neural ensemble activity limit the accuracy of hippocampal spatial
representations. *Nature Communications* 13, 4276.
[doi:10.1038/s41467-022-31254-y](https://doi.org/10.1038/s41467-022-31254-y)

**Dataset.** CRCNS hc-30, [doi:10.6080/K0GH9G5X](https://doi.org/10.6080/K0GH9G5X), file
`hc-30/live/event_traces.mat` streamed from the public S3 bucket and checked against its MD5
(version `unversioned`, as in the other two notebooks, whose headers give the details).

**Why this notebook.** The decoding notebook fits IMSE(n) = I0 n / (1 + n/N) to 4 neuron subsets
per ensemble size on sizes 1, 25, 50, ... (to fit the time limit), where the paper uses 80
subsets on sizes 1, 10, 20, ... (Fig. 2e legend, "an average of over 80 randomly chosen subsets
of cells"; code decode_series, d_neurons 10). The verifier of round 1 asked whether the claims
built on the fitted N and I0 (saturation-ensemble-size, suppl11b-i0-vs-single-cell-snr,
suppl8-task-performance-and-accuracy) fail because of the reduced repetitions. Running 80
subsets on the paper's grid for all 73 large-data sessions would take about 4 hours here, so
this notebook runs it on a set of sessions chosen before any fit was seen:

- every large-data session of Mouse2011 and Mouse2012, the two animals with the lowest per-animal
  N in the decoding notebook (they set the lower bound of the saturation range, and Mouse2011's
  mean I0 decides suppl11b);
- the large-data session of Mouse2028, the animal with the highest per-animal N, with the
  fewest trials (the cost of a decoder grows with the square of the trial count; its other 14
  sessions would take 10 to 15 minutes each on one process).

For each session it reproduces the decoding notebook's fit (same seeds and settings, so the same
numbers) and fits again to 80 subsets on 1, 10, 20, ... and all neurons, unshuffled only (the
three claims use the unshuffled fit). The claims are tested in the decoding notebook; this one
shows how the fit depends on the number of subsets. Run time about 36 minutes on 4 CPUs.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"  # one thread per process; the decoders run in 3 processes
import multiprocessing as mp
import resource
import time

import numpy as np
import pandas as pd

import hc30

T0 = time.time()
sessions, md5, load_s = hc30.load_sessions()
print(f"streamed and parsed event_traces.mat in {load_s:.0f} s; md5 {md5} "
      f"({'matches' if md5 == hc30.MD5 else 'DIFFERS FROM'} checksums.md5)")

# The decoding notebook's settings, reproduced here
REPS_FULL = 10
REPS_SUB = 4
SIZES = [1] + list(range(25, 500, 25))
C_SVM = 1.0
# The paper's settings
REPS_PAPER = 80   # Fig. 2e legend
STEP_PAPER = 10   # code decode_series, default_opt d_neurons 10

## Data check and choice of sessions

In [ ]:
rows, TENS = [], {}
for i, s in enumerate(sessions):
    T, d, tr = hc30.session_tensor(s)
    rows.append(dict(i=i, mouse=s["mouse"], session=s["session_id"], neurons=T.shape[0],
                     right=int((d == 1).sum()), left=int((d == -1).sum())))
    TENS[i] = (T, d)
info = pd.DataFrame(rows)
info["min_trials"] = info[["right", "left"]].min(axis=1)
info["large"] = (info.neurons > 200) & (info.min_trials > 30)
print(f"sessions {len(info)}, mice {info.mouse.nunique()}, large-data sessions {int(info.large.sum())}")
low = info[info.large & info.mouse.isin(["Mouse2011", "Mouse2012"])]
high = info[info.large & (info.mouse == "Mouse2028")].nsmallest(1, "min_trials")
CHOSEN = list(low.i) + list(high.i)
sel = info.loc[CHOSEN].copy()
sel["clock_err_cm"] = [hc30.clock_check(sessions[i], 0) for i in CHOSEN]
sel["misaligned"] = sel.clock_err_cm > 12  # as in the other notebooks
print(f"{len(CHOSEN)} sessions chosen:")
print(sel[["i", "mouse", "session", "neurons", "right", "left", "clock_err_cm", "misaligned"]]
      .round(1).to_string(index=False))
TENS = {i: TENS[i] for i in CHOSEN}

In [ ]:
# Cost estimate: 4 decoders at 100 neurons per session, times the number of decoders
t_est = 0
rng = np.random.default_rng(0)
for i in CHOSEN:
    T, d = TENS[i]
    t = time.time()
    for _ in range(2):
        idx = hc30.equalize_trials(d, rng)
        sub = rng.choice(T.shape[0], 100, replace=False)
        hc30.decode_once(T[sub][:, :, idx], d[idx], rng, False, C=C_SVM)
    per = (time.time() - t) / 2
    n_paper = len(hc30.ensemble_sizes(T.shape[0], STEP_PAPER)) + 1
    t_est += per * (n_paper * REPS_PAPER + 2 * REPS_FULL + 2 * REPS_SUB * len(SIZES))
print(f"estimated {t_est / 60:.0f} min on one process, about {t_est / 180:.0f} min on 3")

## Decoding runs

In [ ]:
def job(i):
    T, d = TENS[i]
    n = T.shape[0]
    # the decoding notebook's run, same seeds
    full = hc30.decode_full(T, d, seed=20000 + i, reps=REPS_FULL, C=C_SVM)
    sizes4 = [s for s in SIZES if s < n]
    iu4, _ = hc30.decode_series(T, d, seed=30000 + i, sizes=sizes4, reps=REPS_SUB, C=C_SVM)
    sizes4 = np.array(sizes4 + [n])
    imse4 = np.r_[iu4, np.mean(1 / np.array(full["mse_u"]))]
    # the paper's repetitions and grid: 80 subsets at 1, 10, 20, ..., all neurons
    sizes80 = np.array([1] + hc30.ensemble_sizes(n, STEP_PAPER))
    imse80, _ = hc30.decode_series(T, d, seed=60000 + i, sizes=sizes80, reps=REPS_PAPER, C=C_SVM,
                                   shuffled=False)
    return i, sizes4, imse4, sizes80, imse80

t = time.time()
with mp.get_context("fork").Pool(3) as pool:
    RUNS = pool.map(job, sorted(CHOSEN, key=lambda i: -info.neurons[i] * info.min_trials[i] ** 2),
                    chunksize=1)
RUNS = {r[0]: r[1:] for r in RUNS}
print(f"decoding: {(time.time() - t) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Fits: 4 subsets on steps of 25 against 80 subsets on steps of 10

Fit as in the decoding notebook (`hc30.fit_imse`). A fit is called degenerate, as there, when N
is below 0 or above 1e6.

In [ ]:
fit_rows = []
for i in CHOSEN:
    s4, m4, s80, m80 = RUNS[i]
    I04, N4, I04c, N4c = hc30.fit_imse(s4, m4)
    I08, N8, I08c, N8c = hc30.fit_imse(s80, m80)
    fit_rows.append(dict(i=i, mouse=info.mouse[i], neurons=info.neurons[i],
                         misaligned=bool(sel.misaligned[i]),
                         I0_4=I04, N_4=N4, N_4_ci=N4c, I0N_4=I04 * N4,
                         I0_80=I08, N_80=N8, N_80_ci=N8c, I0N_80=I08 * N8))
FIT = pd.DataFrame(fit_rows).set_index("i")
FIT["degenerate_4"] = (FIT.N_4 < 0) | (FIT.N_4 > 1e6)
FIT["degenerate_80"] = (FIT.N_80 < 0) | (FIT.N_80 > 1e6)
pd.set_option("display.width", 160)
print(FIT[["mouse", "neurons", "misaligned", "I0_4", "N_4", "N_4_ci", "I0_80", "N_80", "N_80_ci",
           "degenerate_4", "degenerate_80"]].to_string(float_format=lambda v: f"{v:.4g}"))
print(f"degenerate fits: 4 subsets {int(FIT.degenerate_4.sum())}, 80 subsets "
      f"{int(FIT.degenerate_80.sum())} of {len(FIT)}")
ok = ~FIT.degenerate_4 & ~FIT.degenerate_80
ratio = FIT.N_80[ok] / FIT.N_4[ok]
print(f"sessions with both fits non-degenerate: {int(ok.sum())}; N(80) / N(4) median "
      f"{ratio.median():.2f}, range {ratio.min():.2f} to {ratio.max():.2f}; "
      f"I0(80) / I0(4) median {(FIT.I0_80[ok] / FIT.I0_4[ok]).median():.2f}")
print("median 95% half-interval of N relative to N: 4 subsets "
      f"{(FIT.N_4_ci / FIT.N_4.abs())[ok].median():.2f}, 80 subsets "
      f"{(FIT.N_80_ci / FIT.N_80.abs())[ok].median():.2f}")

Each session's IMSE points (1/cm^2, times 1000) against its fitted curve, for both settings
(the verifier asked for a printed table of the fits).

In [ ]:
def curve(n, I0, N):
    n = np.asarray(n, float)
    return I0 * n / (1 + n / N)

for i in CHOSEN:
    s4, m4, s80, m80 = RUNS[i]
    f = FIT.loc[i]
    print(f"session {i} {info.mouse[i]} {info.session[i]} ({info.neurons[i]} neurons"
          f"{', misaligned' if f.misaligned else ''})")
    for lab, s, m, I0, N in (("4 subsets ", s4, m4, f.I0_4, f.N_4), ("80 subsets", s80, m80, f.I0_80, f.N_80)):
        fit = curve(s, I0, N)
        r2 = 1 - np.sum((m - fit) ** 2) / np.sum((m - m.mean()) ** 2)
        print(f"  {lab}: N = {N:.4g}, R^2 = {r2:.3f}")
        print("    n     ", " ".join(f"{v:6d}" for v in s))
        print("    IMSE  ", " ".join(f"{1000 * v:6.2f}" for v in m))
        print("    fit   ", " ".join(f"{1000 * v:6.2f}" for v in fit))

## Per-animal values that the three claims use

saturation-ensemble-size takes the geometric mean of N over an animal's large-data sessions
(N <= 0 left out, as in the decoding notebook) times sqrt(20) - 1; suppl11b takes the mean I0
over an animal's large-data sessions. Mouse2028 has only 2 of its 15 sessions here, so its value
is shown for that session only.

In [ ]:
def gmean(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x) & (x > 0)]
    return np.exp(np.log(x).mean())

K = np.sqrt(20) - 1
for name, sub in (("all chosen sessions", FIT), ("without misaligned sessions", FIT[~FIT.misaligned])):
    print(name)
    for m in ("Mouse2011", "Mouse2012"):
        x = sub[sub.mouse == m]
        print(f"  {m} ({len(x)} sessions): geometric-mean N, 4 subsets {gmean(x.N_4):.0f} "
              f"(5% slope at {K * gmean(x.N_4):.0f} neurons), 80 subsets {gmean(x.N_80):.0f} "
              f"(5% slope at {K * gmean(x.N_80):.0f}); mean I0, 4 subsets {x.I0_4.mean():.3g}, "
              f"80 subsets {x.I0_80.mean():.3g}")
x = FIT[FIT.mouse == "Mouse2028"]
print("Mouse2028, one session, N: 4 subsets", x.N_4.round(0).tolist(), "; 80 subsets", x.N_80.round(0).tolist())
print(f"total run time {(time.time() - T0) / 60:.1f} min")

**Result: the reduced repetitions do not explain the failures of the three claims.** On the 16
sessions, 80 subsets on steps of 10 give fewer degenerate fits (1 against 2) and narrower
intervals for N (median half-interval 0.18 of N against 0.54), but for the 13 sessions with two
non-degenerate fits N changes by a median factor of 1.09 (0.67 to 2.02) and I0 by 0.96. The fits
that make no sense are those of sessions whose events do not track position, with either setting:
sessions 83 and 84 have flat IMSE curves and give I0 = 241 and 230 with N below 1e-5 with 80
subsets, session 51 a negative N, and session 92 an N below 1. Mouse2011's mean I0 over all 11
sessions is 23.7 with 4 subsets and 42.9 with 80; without its misaligned sessions it is 6.5e-05
and 6.3e-05. Without the misaligned sessions the geometric-mean N goes from 458 to 490 (Mouse2011)
and from 268 to 231 (Mouse2012), a 5% slope at 1,590 to 1,702 and 930 to 801 neurons. The one
Mouse2028 session is a near-linear curve with either setting (N = 1.0e9 and 3.8e4).

## Summary

**Result.**

| Claim | Outcome |
|---|---|
| saturation-ensemble-size | tested in the decoding notebook; this notebook checks its fit |
| suppl11b-i0-vs-single-cell-snr | tested in the decoding notebook; this notebook checks its fit |
| suppl8-task-performance-and-accuracy | tested in the decoding notebook; this notebook checks its fit |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.